# Notebook File Overview
This notebook features two different SQL queries based on the Control cohort we created (No diabetes) along with the parameters we set for the variables we selected from the concept sets and dataset builder step.

**The first SQL query extracts our demographic variables we chose**

**The second SQL query extracts the survey question variables we chose**

This *Control* Cohort should have the following characteristics:
* Age 25-50
* Race: White, Black, & Asian
* Ethnicity: Hispanic Yes or No
* Gender: Male or Female
* Condition: NO Diabetes
* PTSD survey questions: Currently seeing a doctor? & Currently prescribed meds?
* BMI
* LDL Cholesterol
* Systolic blood pressure
* A1C

Each query is autogenerated from the details we provided during the cohort builder and dataset builder phase which creates the tables based on the Concept Sets we selected:
1.	Demographics
2.	Survey
3.  Labs and physical measurements

# SQL Code Chunk to Extract Survey Variables

This SQL query extracts the survey variables we selected in the dataset builder concept sets:

* Survey question 1: Currently seeing a doctor?
* Survey question 2: Currently prescribed meds?

The results are filtered and subsetted by the parameters we chose using the Cohort Builder:

* Age group (25-50)
* Ethnicity (Hispanic Yes/No)
* Race (White, Black, or Asian)
* Gender (Male or Female)
* Condition: NO Diabetes

**The following code chunks perform three key steps:**

1. **Creates the SQL query as a text string** (does not execute the query yet)
2. **Creates the file path** and displays it
3. **Executes the BigQuery export** which runs the query and saves results as a CSV file to the specified path

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "diabetes_control_analysis" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_87666367_survey_sql <- paste("
    SELECT
        answer.person_id,
        answer.question,
        answer.answer  
    FROM
        `ds_survey` answer   
    WHERE
        (
            question_concept_id IN (43528846, 43530361)
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `cb_search_person` p 
                WHERE
                    DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                    AND NOT EXISTS (      SELECT
                        'x'      
                    FROM
                        `death` d      
                    WHERE
                        d.person_id = p.person_id ) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `person` p 
                WHERE
                    ethnicity_concept_id IN (38003564, 38003563) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `person` p 
                WHERE
                    gender_concept_id IN (45878463, 45880669) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `person` p 
                WHERE
                    race_concept_id IN (8527, 8516, 8515) ) 
                AND cb_search_person.person_id NOT IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `cb_criteria` cr       
                            WHERE
                                concept_id IN (201820)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 )) criteria ) )
        )", sep="")


######################################################################
# BUILD ORGANIZED FILE PATH FOR BIGQUERY EXPORT                     #
######################################################################


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
survey_87666367_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "survey_87666367",
  "survey_87666367_*.csv")

#####################################################################
# DISPLAY EXPORT DESTINATION PATH                                   #
# Shows the full Cloud Storage path where files will be saved       #
# Save this path - you'll need it to read the data back into R      #
#####################################################################


message(str_glue('The data will be written to {survey_87666367_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))


####################################################################
# EXECUTE SQL QUERY AND EXPORT DIRECTLY TO CLOUD STORAGE           #
####################################################################


# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_87666367_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  survey_87666367_path,
  destination_format = "CSV")



## Finally, this chunk creates a function that reads the exported CSV file back from our *workspace bucket* into R as a single *data frame* (i.e. survey_87666367_df) stored in R session memory.

This gives us a workable data frame that we can:
* Manipulate and clean
* Analyze and summarize
* Work with other R data frames we create

In [ ]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {survey_87666367_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(survey = col_character(), question = col_character(), answer = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_87666367_survey_df <- read_bq_export_from_workspace_bucket(survey_87666367_path)

dim(dataset_87666367_survey_df)

head(dataset_87666367_survey_df, 5)

# SQL Code Chunk to Extract Measurement Variables

This SQL query extracts the Measurements variables we selected in the dataset builder concept sets:

* BMI
* LDL Cholesterol
* Systolic blood pressure
* A1C

The results are filtered and subsetted by the parameters we chose using the Cohort Builder:

* Age group (25-50)
* Ethnicity (Hispanic Yes/No)
* Race (White, Black, or Asian)
* Gender (Male or Female)
* Condition: NO Diabetes

**The following code chunks perform three key steps:**

1. **Creates the SQL query as a text string** (does not execute the query yet)
2. **Creates the file path** and displays it
3. **Executes the BigQuery export** which runs the query and saves results as a CSV file to the specified path

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "diabetes_control_analysis" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_87666367_measurement_sql <- paste("
      SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        measurement.measurement_datetime,
        measurement.value_as_number 
    FROM
        ( SELECT
            * 
        FROM
            `measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (3004249, 3004410, 3038553, 40795800, 4152194)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                        AND NOT EXISTS (      SELECT
                            'x'      
                        FROM
                            `death` d      
                        WHERE
                            d.person_id = p.person_id ) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `person` p 
                    WHERE
                        ethnicity_concept_id IN (38003564, 38003563) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `person` p 
                    WHERE
                        gender_concept_id IN (45878463, 45880669) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `person` p 
                    WHERE
                        race_concept_id IN (8527, 8516, 8515) ) 
                    AND cb_search_person.person_id NOT IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
            )) measurement 
    LEFT JOIN
        `concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
measurement_87666367_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "measurement_87666367",
  "measurement_87666367_*.csv")
message(str_glue('The data will be written to {measurement_87666367_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_87666367_measurement_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  measurement_87666367_path,
  destination_format = "CSV")



## REMINDER: this chunk creates a function that reads the exported CSV file back from our *workspace bucket* into R as a single *data frame* 

This gives us a workable data frame that we can:
* Manipulate and clean
* Analyze and summarize
* Work with other R data frames we create

In [ ]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {measurement_87666367_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_87666367_measurement_df <- read_bq_export_from_workspace_bucket(measurement_87666367_path)

dim(dataset_87666367_measurement_df)

head(dataset_87666367_measurement_df, 5)

# SQL Code Chunk to Extract Demographic Variables

This part is an SQL query which extracts the demographic variables we selected in the dataset builder filtered and subsetted by the parameters we chose using the Cohort Builder:

* Age-group (25-50)
* Ethnicity (Hispanic Yes/No)
* Race (White, Black, or Asian)
* Gender (Male or Female)
* Inclusion criteria (we EXCLUDE those with diabetes)

**The following code chunks perform three key steps:**

1. **Creates the SQL query as a text string** (does not execute the query yet)
2. **Creates the file path** and displays it
3. **Executes the BigQuery export** which runs the query and saves results as a CSV file to the specified path

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "diabetes_control_analysis" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_87666367_person_sql <- paste("
      SELECT
        person.person_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        p_race_concept.concept_name as race,
        p_ethnicity_concept.concept_name as ethnicity,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                AND NOT EXISTS (      SELECT
                    'x'      
                FROM
                    `death` d      
                WHERE
                    d.person_id = p.person_id ) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `person` p 
            WHERE
                ethnicity_concept_id IN (38003564, 38003563) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `person` p 
            WHERE
                gender_concept_id IN (45878463, 45880669) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `person` p 
            WHERE
                race_concept_id IN (8527, 8516, 8515) ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `cb_criteria` cr       
                        WHERE
                            concept_id IN (201820)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_87666367_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person_87666367",
  "person_87666367_*.csv")
message(str_glue('The data will be written to {person_87666367_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_87666367_person_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  person_87666367_path,
  destination_format = "CSV")



In [ ]:
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_87666367_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(gender = col_character(), race = col_character(), ethnicity = col_character(), sex_at_birth = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_87666367_person_df <- read_bq_export_from_workspace_bucket(person_87666367_path)

dim(dataset_87666367_person_df)

head(dataset_87666367_person_df, 5)

# Run the provided snippets to save our 3 raw datasets to our workspace bucket

This section saves R data frames that are already in memory to our workspace bucket using a two-step process:

1. Writes dataframe to local CSV file on persistent disk
2. Copies file from local storage to workspace bucket using gsutil

It also verifies successful upload by listing bucket contents

Once these datasets are saved to the workspace bucket, we no longer need our persistent disk

**Please note that we will be updating this code ourselves and even chaging the code after the warning**

<div style=" background-color:#f8d7da; color:#842029; border:1px solid #f5c2c7; border-radius:8px; padding:20px; font-size:20px; "> 
<b>We are going to update the provided snippet, even past the <i>DON'T CHANGE FROM HERE</i> Warning</b><br> 
</div>

In [ ]:
# Setup snippet
library(tidyverse)  # Data wrangling packages.

In [ ]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
person_control <- dataset_87666367_person_df
survey_control <- dataset_87666367_survey_df
measurement_control <- dataset_87666367_measurement_df

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
person_control_filename <- 'person_control.csv'
survey_control_filename <- 'survey_control.csv'
measurement_control_filename <- 'measurement_control.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(person_control, person_control_filename)
write_excel_csv(survey_control, survey_control_filename)
write_excel_csv(measurement_control, measurement_control_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", person_control_filename, " ", my_bucket, "/data/"), intern=T)
system(paste0("gsutil cp ./", survey_control_filename, " ", my_bucket, "/data/"), intern=T)
system(paste0("gsutil cp ./", measurement_control_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)


<div class="alert alert-block alert-danger" style="font-size:24px; padding: 10px;">
    <b>Once you save to the workspace bucket, you do not need to run the queries again</b>
    <ul style="font-size:20px; margin-top: 5px;">
        <li>Recommend using a new Jupyter Notebook to read in saved data for analysis</li>
    </ul>
</div>